In [1]:
import torch
import torch.nn as nn

# Single
from mpramnist.Agarwal2025.dataset import AgarwalSingleDataset
from mpramnist.Agarwal2025.trainer import LitModel_AgarwalSingle

# Multi
from mpramnist.Agarwal2025.dataset import AgarwalMultiDataset
from mpramnist.Agarwal2025.trainer import LitModel_AgarwalMulti

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights
import mpramnist.transforms as t

from torch.utils.data import DataLoader

import lightning.pytorch as L

In [2]:
constant_left_flank = AgarwalSingleDataset.CONSTANT_LEFT_FLANK  # required for each sequence
constant_rigtht_flank = (AgarwalSingleDataset.CONSTANT_RIGHT_FLANK)  # required for each sequence
left_flank = AgarwalSingleDataset.LEFT_FLANK  # original flanks from human_legnet
right_flank = AgarwalSingleDataset.RIGHT_FLANK

In [3]:
# preprocessing
train_transform = t.Compose(
    [
        t.AddFlanks(constant_left_flank, constant_rigtht_flank),
        t.AddFlanks("", right_flank),  # this is original parameters for human_legnet
        t.RightCrop(230, 260),  # this is using for shifting
        t.LeftCrop(230, 230),
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(constant_left_flank, constant_rigtht_flank),
        t.ReverseComplement(0),
        t.Seq2Tensor(),
    ]
)

In [4]:
from torchmetrics import PearsonCorrCoef

forw_transform = t.Compose(
    [t.AddFlanks(constant_left_flank, constant_rigtht_flank), t.Seq2Tensor()]
)
rev_transform = t.Compose(
    [
        t.AddFlanks(constant_left_flank, constant_rigtht_flank),
        t.ReverseComplement(1),
        t.Seq2Tensor(),
    ]
)


def meaned_prediction(forw, rev, trainer, seq_model, name, out_channels):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef(num_outputs=out_channels)
    print(name, " Pearson correlation")

    return pears(mean_forw, targets)

# AgarwalSingle

In [5]:
# Read the MPRAdata, preprocess them and encapsulate them into dataloader form.
Cell_Type = "HepG2" # or K562 or WTC11
train_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="train", transform=train_transform, root="../data/",
)

val_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="val", transform=test_transform, root="../data/",
)

test_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="test", transform=test_transform, root="../data/",
)

# encapsulate data into dataloader form
train_loader = DataLoader(
    dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=103
)

val_loader = DataLoader(
    dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=103
)

test_loader = DataLoader(
    dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=103
)

in_channels = len(train_dataset[0][0])
out_channels = 1

In [6]:
print(train_dataset)

Dataset AgarwalSingleDataset (MpraDaraset)
    Number of datapoints: 98336
    Root location: ../data/Agarwal
    Using split: [1, 2, 3, 4, 5, 6, 7, 8]
    Split: {'train': 98336, 'val': 12298, 'test': 12298}
    Task: Regression
    Description: The AgarwalSingle dataset is based on a lentiviral MPRA system. The total data volume was 122,926 sequences for the HepG2 cell line, 196,664 for K562, and 46,185 for WTC11. Each sequence is 200 nucleotides long. The regression task was to predict a scalar value of regulatory activity for the corresponding cell line.


In [7]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_HepG2 = LitModel_AgarwalSingle(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=10
)

In [ ]:
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=50,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

In [ ]:
# Train the model
trainer.fit(seq_model_HepG2, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model_HepG2, dataloaders=test_loader)

In [ ]:
test_forw = AgarwalSingleDataset(cell_type=Cell_Type, split="test", transform=forw_transform, root="../data/")
test_rev = AgarwalSingleDataset(cell_type=Cell_Type, split="test", transform=rev_transform, root="../data/")

forw_hepg2 = DataLoader(dataset=test_forw, batch_size=1024, shuffle=False, num_workers=103, pin_memory=True,)
rev_hepg2 = DataLoader(dataset=test_rev, batch_size=1024, shuffle=False, num_workers=103, pin_memory=True,)

meaned_prediction(forw_hepg2, rev_hepg2, trainer, seq_model_HepG2, "HepG2")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


HepG2 Pearson correlation


tensor(0.8039)

# AgarwalMulti

In [8]:
cell_types = ["HepG2", "K562", "WTC11"]

train_dataset = AgarwalMultiDataset(
    cell_type=cell_types, split="train", transform=train_transform, root="../data/",
)

val_dataset = AgarwalMultiDataset(
    cell_type=cell_types, split="val", transform=test_transform, root="../data/",
)

test_dataset = AgarwalMultiDataset(
    cell_type=cell_types, split="test", transform=test_transform, root="../data/",
)

# encapsulate data into dataloader form
train_loader = DataLoader(
    dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=103
)

val_loader = DataLoader(
    dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=103
)

test_loader = DataLoader(
    dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=103
)

in_channels = len(train_dataset[0][0])
out_channels = len(cell_types)

In [9]:
print(train_dataset)

Dataset AgarwalMultiDataset (MpraDaraset)
    Number of datapoints: 44264
    Root location: ../data/AgarwalJoint
    Using split: [1, 2, 3, 4, 5, 6, 7, 8]
    Split: {'train': 36946, 'val': 4622, 'test': 4622}
    Task: Regression
    Description: The AgarwalMulti (joint) dataset contains 55,338 sequences, each 200 nucleotides long, comprising enhancers from the HepG2, K562, and WTC11 cell lines. The regression task is to predict three scalar values of regulatory activity for each of these cell lines.


In [10]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_multi = LitModel_AgarwalMulti(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=10
)

In [ ]:
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

In [ ]:
# Train the model
trainer.fit(seq_model_multi, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model_multi, dataloaders=test_loader)

In [ ]:
test_forw = AgarwalMultiDataset(cell_type=cell_types, split="test", transform=forw_transform, root="../data/")
test_rev = AgarwalMultiDataset(cell_type=cell_types, split="test", transform=rev_transform, root="../data/")

forw_multi = DataLoader(dataset=test_forw, batch_size=1024, shuffle=False, num_workers=103, pin_memory=True,)
rev_multi = DataLoader(dataset=test_rev, batch_size=1024, shuffle=False, num_workers=103, pin_memory=True,)

meaned_prediction(forw_multi, rev_multi, trainer, seq_model_multi, cell_types, out_channels)